In [1]:
import torch
import torch.nn as nn
import numpy as np
from model.model import MelNet
from model.loss import GMMLoss
from torch.utils.data import DataLoader
from utils.data import MelData, MelCollate
from utils.utils import *
import utils.hparams as hp
import matplotlib.pyplot as plt
%matplotlib inline

use_GMM=False
dataset = MelData(hp)
collate_fn = MelCollate(hp)
dataloader = DataLoader(dataset, batch_size = hp.batch_size, shuffle=True, drop_last=True, collate_fn = collate_fn)

In [2]:
device = torch.device('cuda:0')
model = MelNet(hp).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-6)
criterion = GMMLoss() if use_GMM else nn.L1Loss()

In [3]:
model.eval()
with torch.no_grad():
    pred = model.sample(1, device=device)

0
1
2
3
4
5
6
7
8
9
10


KeyboardInterrupt: 

In [ ]:
print("Training Start!!!")
iteration=1
for epoch in range(hp.epochs):
    for tiers in dataloader:
        
        lr_scheduling(optimizer, iteration, 3e-4, 10)
        
        model.zero_grad()
        mu, std, pi = model(tiers[1].to(device), 1)

        if use_GMM:
            loss = criterion(tiers[1].to(device), mu, std, pi)
        else:
            loss = criterion(tiers[1].to(device), mu[:,:,:,0])
            
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        iteration += 1
        
        if iteration%100==0:
            fig = plt.figure(figsize=(20,10))
            ax1, ax2, ax3 = fig.add_subplot(221), fig.add_subplot(222), fig.add_subplot(223)
            ax1.imshow(tiers[1][0].numpy(), origin='lower')
            ax2.imshow(mu[0,:,:,0].cpu().detach().numpy(), origin='lower')
            ax3.imshow(np.tile(mu[0,:,:,0][0:1].cpu().detach().numpy(), (16,1)), origin='lower')
            plt.show()
            print(mu[0,:,:,0][0].cpu().detach().numpy()[:10])
            print(mu[0,:,:,0][1].cpu().detach().numpy()[:10])
            print(loss.item())
    break